# xsmith walkthrough

One target. One pass through the pipeline. Requires `ANTHROPIC_API_KEY`.


In [2]:
import sys, os, pathlib, asyncio
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# env = ROOT / ".env"
# if env.exists():
#     for line in env.read_text().splitlines():
#         if "=" in line and not line.strip().startswith("#"):
#             k, v = line.split("=", 1)
#             os.environ.setdefault(k.strip(), v.strip())

# assert os.environ.get("ANTHROPIC_API_KEY"), "set ANTHROPIC_API_KEY"


## 1. Target module


In [ ]:
from xsmith.domain.target import Target

SOURCE = '''
def classify(n):
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
    if n % 2 == 0:
        return "positive even"
    return "positive odd"
'''.lstrip()

target = Target(target_id="demo/classify", module_path="demo.classify", source=SOURCE)


## 2. Discover branch universe

Run an import-only test under coverage.py to learn every branch arc in the module.


In [5]:
from xsmith.execution.subprocess_runner import SubprocessRunner

runner = SubprocessRunner(timeout_s=15.0)
universe = await runner.discover_branches(target)
target.branches = universe
print(len(universe), "branches")
for b in sorted(universe, key=lambda b: (b.src, b.dst)):
    print(" ", b.key())


6 branches
  target_pkg/classify.py:2->3
  target_pkg/classify.py:2->4
  target_pkg/classify.py:4->5
  target_pkg/classify.py:4->6
  target_pkg/classify.py:6->7
  target_pkg/classify.py:6->8


## 3. Initial coverage state


In [6]:
from xsmith.domain.coverage import CoverageMap

cov = CoverageMap(total=universe)
print(f"{len(cov.covered)}/{len(cov.total)} covered")


0/6 covered


## 4. One generator call (variant 1 = edge cases)


In [ ]:
from xsmith.agents.test_generator import TestGeneratorAgent

gen = TestGeneratorAgent(variant_idx=1, model="claude-sonnet-4-6", max_turns=6)
candidate, gen_usage = await gen.propose(target=target, uncovered=cov.uncovered, history=[])

print("rationale:", candidate.rationale)
print("---")
print(candidate.code)
print(f"cost: ${gen_usage.cost_usd:.4f}, in={gen_usage.tokens_in}, out={gen_usage.tokens_out}")


## 5. One scorer call on that candidate


In [ ]:
from xsmith.agents.scorer import QValueScorerAgent

scorer = QValueScorerAgent(model="claude-sonnet-4-6", max_turns=3, gamma=0.5)
score, score_usage = await scorer.score(target=target, uncovered=cov.uncovered, candidate=candidate)
print(f"Q = {score.immediate_branches} + {score.gamma} * {score.future_value} = {score.q}")


## 6. Execute the candidate, update coverage


In [ ]:
run_result = await runner.run(candidate, target)
new = cov.update(run_result.branches_covered)
print(f"outcome={run_result.outcome}, +{len(new)} new branches, cov={len(cov.covered)}/{len(cov.total)}")
if run_result.stderr.strip():
    print("stderr:", run_result.stderr[:400])


## 7. Full strategy step (K=5 parallel generators + scoring + argmax)

This is what one iteration of `ExplorationLoop` does internally.


In [ ]:
from xsmith.strategies.cov_qvalue import CovQValueStrategy

strategy = CovQValueStrategy(model="claude-sonnet-4-6", k=5, gamma=0.5)
winner, usage = await strategy.propose(target=target, coverage=cov, history=[])
print(winner.rationale)
print("---")
print(winner.code)
print(f"aggregate cost: ${usage.cost_usd:.4f}")


## 8. Drive the loop for a small budget


In [ ]:
from xsmith.domain.budget import Budget
from xsmith.exploration.explorer import ExplorationLoop

cov = CoverageMap(total=universe)  # reset
budget = Budget(exec_remaining=3)
loop = ExplorationLoop(strategy=strategy, runner=runner)
result = await loop.explore(target=target, budget=budget, initial_coverage=cov)

for it in result.iterations:
    print(f"iter {it.iteration}: {it.run_result.outcome:>5}  +{len(it.new_branches)} new  cov={it.coverage_after}/{it.coverage_total}  ${it.agent_usage.cost_usd:.4f}")
print(f"final: {result.covered_count}/{result.total_count} ({result.coverage_fraction:.0%})")
